# LGN `__init__` Numba/Numpy Benchmark

This notebook defines `LGNNumbaNumpyInit`, a drop-in LGN class variant where one-time `__init__` preprocessing is moved from TensorFlow ops to Numba/Numpy.

It runs:
- Attribute-level equivalence checks vs `lgn_model.lgn.LGN`
- Functional equivalence checks (`spatial_response`, `firing_rates_from_spatial`)
- Init-time benchmark (warm cache)
- Init-time benchmark (cold spatial-cache path)


In [1]:
import gc
import os
import pickle as pkl
import shutil
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

import lgn_model.lgn as lgn_module

try:
    from numba import njit
    HAS_NUMBA = True
except Exception:
    HAS_NUMBA = False

print(f"TensorFlow: {tf.__version__}")
print(f"Numba available: {HAS_NUMBA}")


/tmp/ipykernel_78553/3449165710.py:10: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
2026-02-12 15:11:40.590210: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-12 15:11:40.627180: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-12 15:1

TensorFlow: 2.15.0
Numba available: True


In [2]:
# Config
ROW_SIZE = 80
COL_SIZE = 120
N_INPUT = 17400
DTYPE = tf.float32

data_dir_candidates = [
    # 'GLIF_network',
    # 'GLIF_network_nll',
    'GLIF_network_nll_full',
    # 'GLIF_network_nll_core',
]

repo_root = Path(lgn_module.__file__).resolve().parent.parent
DATA_DIR = None
for candidate in data_dir_candidates:
    p = Path(candidate)
    if p.is_absolute() and p.is_dir():
        DATA_DIR = str(p)
        break
    p_repo = repo_root / candidate
    if p_repo.is_dir():
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(f"No valid data dir found. Tried: {data_dir_candidates}")

print(f"Using data_dir={DATA_DIR}")


Using data_dir=GLIF_network_nll_full


In [3]:
if HAS_NUMBA:
    @njit(cache=True)
    def _assign_spatial_bin_ids_numba(spatial_sizes, spatial_range):
        """Assign each spatial size to a [low, high) range bin index, or -1 if none."""
        n = spatial_sizes.shape[0]
        n_bins = spatial_range.shape[0] - 1
        out = np.full(n, -1, dtype=np.int32)
        for idx in range(n):
            v = spatial_sizes[idx]
            for b in range(n_bins):
                if (v >= spatial_range[b]) and (v < spatial_range[b + 1]):
                    out[idx] = b
                    break
        return out
else:
    def _assign_spatial_bin_ids_numba(spatial_sizes, spatial_range):
        n_bins = len(spatial_range) - 1
        out = np.full(spatial_sizes.shape[0], -1, dtype=np.int32)
        for b in range(n_bins):
            sel = np.logical_and(spatial_sizes >= spatial_range[b], spatial_sizes < spatial_range[b + 1])
            out[sel] = b
        return out


In [4]:
class LGNNumbaNumpyInit(object):
    """
    Drop-in LGN class with __init__ preprocessing moved to Numba/Numpy.
    Runtime methods are reused from the original class.
    """

    spatial_response = lgn_module.LGN.spatial_response
    firing_rates_from_spatial = lgn_module.LGN.firing_rates_from_spatial

    def __init__(self, row_size=80, col_size=120, data_dir='GLIF_network', n_input=None, dtype=tf.float32):
        filename = f'lgn_full_col_cells_{col_size}x{row_size}.csv'
        lgn_code_dir = os.path.split(lgn_module.__file__)[0]
        root_dir = os.path.split(lgn_code_dir)[0]
        if os.path.isabs(data_dir):
            data_dir_abs = data_dir
        else:
            data_dir_abs = os.path.join(root_dir, data_dir)

        lgn_data_dir = os.path.join(data_dir_abs, 'tf_data')
        lgn_data_path = os.path.join(lgn_data_dir, filename)

        if os.path.exists(lgn_data_path):
            d = pd.read_csv(lgn_data_path, delimiter=' ')
        else:
            os.makedirs(lgn_data_dir, exist_ok=True)
            network_dir = os.path.join(data_dir_abs, 'network')
            lgn_node_path = os.path.join(network_dir, 'lgn_nodes.h5')
            lgn_node_type_path = os.path.join(network_dir, 'lgn_node_types.csv')
            d = lgn_module.create_lgn_units_info(
                filename=lgn_data_path,
                csv_path=lgn_node_type_path,
                h5_path=lgn_node_path,
            )

        if n_input is not None and n_input < len(d):
            d = d.iloc[:n_input].copy()

        n_units = len(d)
        s_path = os.path.join(lgn_data_dir, f'spontaneous_firing_rates_{col_size}x{row_size}_n{n_units}.pkl')
        t_path = os.path.join(lgn_data_dir, f'temporal_kernels_{col_size}x{row_size}_n{n_units}.pkl')
        spatial_path = os.path.join(lgn_data_dir, f'spatial_kernels_{col_size}x{row_size}_n{n_units}.pkl')

        self.dtype = dtype
        model_id = d['model_id'].to_numpy()
        amplitude = np.array([1.0 if m.count('ON') > 0 else -1.0 for m in model_id], dtype=np.float32)
        non_dom_amplitude = np.zeros_like(amplitude)
        is_composite = np.array([('ON' in m and 'OFF' in m) for m in model_id], dtype=np.float32)

        if not os.path.exists(s_path):
            cell_type = [a[:a.find('_')] for a in model_id]
            tf_str = [a[a.find('_') + 1:] for a in model_id]
            spontaneous_firing_rates = []
            for a, b in zip(cell_type, tf_str):
                if a.count('ON') > 0 and a.count('OFF') > 0:
                    spontaneous_firing_rates.append(-1.0)
                else:
                    spontaneous_firing_rate = lgn_module.get_data_metrics_for_each_subclass(a)[b]['spont_exp']
                    spontaneous_firing_rates.append(spontaneous_firing_rate[0])
            spontaneous_firing_rates = np.array(spontaneous_firing_rates, dtype=np.float32)
            with open(s_path, 'wb') as f:
                pkl.dump(spontaneous_firing_rates, f)
        else:
            with open(s_path, 'rb') as f:
                spontaneous_firing_rates = np.asarray(pkl.load(f), dtype=np.float32)

        if not os.path.exists(t_path):
            nkt = 600
            kernel_length = 700
            dom_temporal_kernels = []
            non_dom_temporal_kernels = []

            tuning_angle = d['tuning_angle'].to_numpy(dtype=np.float32)
            subfield_separation = d['sf_sep'].to_numpy(dtype=np.float32)
            x = d['x'].to_numpy(dtype=np.float32)
            y = d['y'].to_numpy(dtype=np.float32)
            non_dominant_x = np.zeros_like(x)
            non_dominant_y = np.zeros_like(y)

            temporal_peaks_dom = np.stack(
                (d['kpeaks_dom_0'].to_numpy(dtype=np.float32), d['kpeaks_dom_1'].to_numpy(dtype=np.float32)),
                axis=-1,
            )
            temporal_weights = np.stack(
                (d['weight_dom_0'].to_numpy(dtype=np.float32), d['weight_dom_1'].to_numpy(dtype=np.float32)),
                axis=-1,
            )
            temporal_delays = np.stack(
                (d['delay_dom_0'].to_numpy(dtype=np.float32), d['delay_dom_1'].to_numpy(dtype=np.float32)),
                axis=-1,
            )

            temporal_peaks_non_dom = np.stack(
                (d['kpeaks_non_dom_0'].to_numpy(dtype=np.float32), d['kpeaks_non_dom_1'].to_numpy(dtype=np.float32)),
                axis=-1,
            )
            temporal_weights_non_dom = np.stack(
                (d['weight_non_dom_0'].to_numpy(dtype=np.float32), d['weight_non_dom_1'].to_numpy(dtype=np.float32)),
                axis=-1,
            )
            temporal_delays_non_dom = np.stack(
                (d['delay_non_dom_0'].to_numpy(dtype=np.float32), d['delay_non_dom_1'].to_numpy(dtype=np.float32)),
                axis=-1,
            )

            for i in range(x.shape[0]):
                dom_temporal_kernel = np.zeros((kernel_length,), np.float32)
                non_dom_temporal_kernel = np.zeros((kernel_length,), np.float32)
                if model_id[i].count('ON') > 0 and model_id[i].count('OFF') > 0:
                    non_dom_params = dict(
                        opt_wts=temporal_weights_non_dom[i],
                        opt_kpeaks=temporal_peaks_non_dom[i],
                        opt_delays=temporal_delays_non_dom[i],
                    )
                    dom_params = dict(
                        opt_wts=temporal_weights[i],
                        opt_kpeaks=temporal_peaks_dom[i],
                        opt_delays=temporal_delays[i],
                    )
                    amp_on = 1.0

                    if model_id[i].count('sONsOFF_001') > 0:
                        non_dom_filter, non_dom_sum = lgn_module.create_one_unit_of_two_subunit_filter(non_dom_params, 121.0)
                        dom_filter, dom_sum = lgn_module.create_one_unit_of_two_subunit_filter(dom_params, 115.0)
                        spont = 4.0
                        max_roff = 35.0
                        max_ron = 21.0
                        amp_off = -(max_roff / max_ron) * (non_dom_sum / dom_sum) * amp_on - (
                            spont * (max_roff - max_ron)
                        ) / (max_ron * dom_sum)
                    elif model_id[i].count('sONtOFF_001') > 0:
                        non_dom_filter, non_dom_sum = lgn_module.create_one_unit_of_two_subunit_filter(non_dom_params, 93.5)
                        dom_filter, dom_sum = lgn_module.create_one_unit_of_two_subunit_filter(dom_params, 64.8)
                        spont = 5.5
                        max_roff = 46.0
                        max_ron = 31.0
                        amp_off = -0.7 * (max_roff / max_ron) * (non_dom_sum / dom_sum) * amp_on - (
                            spont * (max_roff - max_ron)
                        ) / (max_ron * dom_sum)
                    else:
                        raise ValueError('Unknown cell type')

                    non_dom_amplitude[i] = amp_on
                    amplitude[i] = amp_off
                    spontaneous_firing_rates[i] = spont / 2.0

                    hor_offset = np.cos(tuning_angle[i] * np.pi / 180.0) * subfield_separation[i] + x[i]
                    vert_offset = np.sin(tuning_angle[i] * np.pi / 180.0) * subfield_separation[i] + y[i]
                    non_dominant_x[i] = hor_offset
                    non_dominant_y[i] = vert_offset
                    dom_temporal_kernel[-len(dom_filter.kernel_data):] = dom_filter.kernel_data[::-1]
                    non_dom_temporal_kernel[-len(non_dom_filter.kernel_data):] = non_dom_filter.kernel_data[::-1]
                else:
                    dd = dict(
                        neye=0,
                        ncos=2,
                        kpeaks=temporal_peaks_dom[i],
                        b=0.3,
                        delays=[temporal_delays[i].astype(int)],
                    )
                    kernel_data = np.dot(lgn_module.makeBasis_StimKernel(dd, nkt), temporal_weights[i])
                    dom_temporal_kernel[-len(kernel_data):] = kernel_data

                dom_temporal_kernels.append(dom_temporal_kernel)
                non_dom_temporal_kernels.append(non_dom_temporal_kernel)

            dom_temporal_kernels = np.asarray(dom_temporal_kernels, dtype=np.float32)
            non_dom_temporal_kernels = np.asarray(non_dom_temporal_kernels, dtype=np.float32)

            dom_cumsum = np.cumsum(np.abs(dom_temporal_kernels), axis=1)
            non_dom_cumsum = np.cumsum(np.abs(non_dom_temporal_kernels), axis=1)
            threshold = 1e-6
            dom_truncation_points = np.sum(dom_cumsum <= threshold, axis=1)
            non_dom_truncation_points = np.where(
                np.sum(np.abs(non_dom_temporal_kernels), axis=1) > 0,
                np.sum(non_dom_cumsum <= threshold, axis=1),
                np.inf,
            )

            dom_truncation = int(np.min(dom_truncation_points))
            if np.all(np.isinf(non_dom_truncation_points)):
                non_dom_truncation = dom_truncation
            else:
                non_dom_truncation = int(np.min(non_dom_truncation_points))

            truncation = int(np.min([dom_truncation, non_dom_truncation]))
            dom_temporal_kernels = dom_temporal_kernels[:, dom_truncation:].T
            non_dom_temporal_kernels = non_dom_temporal_kernels[:, non_dom_truncation:].T
            print(f'Kernels truncated from time step {truncation} onwards.')

            to_save = dict(
                dom_temporal_kernels=dom_temporal_kernels,
                non_dom_temporal_kernels=non_dom_temporal_kernels,
                non_dominant_x=non_dominant_x,
                non_dominant_y=non_dominant_y,
                amplitude=amplitude.astype(np.float32),
                non_dom_amplitude=non_dom_amplitude.astype(np.float32),
                spontaneous_firing_rates=np.asarray(spontaneous_firing_rates, dtype=np.float32),
            )
            with open(t_path, 'wb') as f:
                pkl.dump(to_save, f)
        else:
            with open(t_path, 'rb') as f:
                loaded = pkl.load(f)
            dom_temporal_kernels = np.asarray(loaded['dom_temporal_kernels'], dtype=np.float32)
            non_dom_temporal_kernels = np.asarray(loaded['non_dom_temporal_kernels'], dtype=np.float32)
            non_dominant_x = np.asarray(loaded['non_dominant_x'], dtype=np.float32)
            non_dominant_y = np.asarray(loaded['non_dominant_y'], dtype=np.float32)
            amplitude = np.asarray(loaded['amplitude'], dtype=np.float32)
            non_dom_amplitude = np.asarray(loaded['non_dom_amplitude'], dtype=np.float32)
            spontaneous_firing_rates = np.asarray(loaded['spontaneous_firing_rates'], dtype=np.float32)
            x = d['x'].to_numpy(dtype=np.float32)
            y = d['y'].to_numpy(dtype=np.float32)

        if not os.path.exists(spatial_path):
            col_max = float(col_size - 1)
            row_max = float(row_size - 1)

            x = d['x'].to_numpy(dtype=np.float32)
            y = d['y'].to_numpy(dtype=np.float32)
            x = np.clip(x * col_max / col_size, 0, col_max)
            y = np.clip(y * row_max / row_size, 0, row_max)

            non_dominant_x = np.clip(non_dominant_x * col_max / col_size, 0, col_max)
            non_dominant_y = np.clip(non_dominant_y * row_max / row_size, 0, row_max)

            d_spatial = 1.0
            spatial_range = np.arange(0, 15, d_spatial, dtype=np.float32)
            x_range = np.arange(-50, 51)
            y_range = np.arange(-50, 51)
            spatial_sizes = d['spatial_size'].to_numpy(dtype=np.float32)

            bin_ids = _assign_spatial_bin_ids_numba(spatial_sizes, spatial_range)
            gaussian_filters = []
            spatial_range_indices = []

            for i in range(len(spatial_range) - 1):
                indices = np.where(bin_ids == i)[0].astype(np.int32)
                if indices.size == 0:
                    continue

                spatial_range_indices.append(indices)
                sigma = (spatial_range[i] + d_spatial / 2.0) / 3.0
                original_filter = lgn_module.GaussianSpatialFilter(
                    translate=(0.0, 0.0), sigma=(sigma, sigma), origin=(0.0, 0.0)
                )
                kernel = original_filter.get_kernel(x_range, y_range, amplitude=1.0).full()
                nonzero_inds = np.where(np.abs(kernel) > 1e-9)
                rm, rM = nonzero_inds[0].min(), nonzero_inds[0].max()
                cm, cM = nonzero_inds[1].min(), nonzero_inds[1].max()
                kernel = kernel[rm:rM + 1, cm:cM + 1]
                gaussian_filter = kernel[..., None, None].astype(np.float32, copy=False)
                gaussian_filters.append(gaussian_filter)

            if len(spatial_range_indices) > 0:
                neuron_ids = np.concatenate(spatial_range_indices, axis=0).astype(np.int32, copy=False)
                sorted_neuron_ids_indices = np.argsort(neuron_ids).astype(np.int32, copy=False)
            else:
                sorted_neuron_ids_indices = np.empty((0,), dtype=np.int32)

            to_save = dict(
                x=x,
                y=y,
                non_dominant_x=non_dominant_x,
                non_dominant_y=non_dominant_y,
                gaussian_filters=gaussian_filters,
                spatial_range_indices=spatial_range_indices,
                sorted_neuron_ids_indices=sorted_neuron_ids_indices,
            )
            with open(spatial_path, 'wb') as f:
                pkl.dump(to_save, f)
        else:
            with open(spatial_path, 'rb') as f:
                loaded = pkl.load(f)
            x = np.asarray(loaded['x'], dtype=np.float32)
            y = np.asarray(loaded['y'], dtype=np.float32)
            non_dominant_x = np.asarray(loaded['non_dominant_x'], dtype=np.float32)
            non_dominant_y = np.asarray(loaded['non_dominant_y'], dtype=np.float32)
            gaussian_filters = [np.asarray(gf, dtype=np.float32) for gf in loaded['gaussian_filters']]
            spatial_range_indices = [np.asarray(a, dtype=np.int32) for a in loaded['spatial_range_indices']]
            sorted_neuron_ids_indices = np.asarray(loaded['sorted_neuron_ids_indices'], dtype=np.int32)

        self.x = tf.constant(x, dtype=dtype)
        self.y = tf.constant(y, dtype=dtype)
        self.non_dominant_x = tf.constant(non_dominant_x, dtype=dtype)
        self.non_dominant_y = tf.constant(non_dominant_y, dtype=dtype)
        self.amplitude = tf.constant(amplitude, dtype=dtype)
        self.non_dom_amplitude = tf.constant(non_dom_amplitude, dtype=dtype)
        self.is_composite = tf.constant(is_composite, dtype=dtype)
        self.spontaneous_firing_rates = tf.constant(spontaneous_firing_rates, dtype=dtype)

        self.dom_temporal_kernels = tf.convert_to_tensor(dom_temporal_kernels, dtype=dtype)
        self.non_dom_temporal_kernels = tf.convert_to_tensor(non_dom_temporal_kernels, dtype=dtype)
        self.gaussian_filters = [tf.convert_to_tensor(gf, dtype=dtype) for gf in gaussian_filters]
        self.spatial_range_indices = spatial_range_indices
        self.sorted_neuron_ids_indices = tf.convert_to_tensor(sorted_neuron_ids_indices, dtype=tf.int32)


In [5]:
def _as_numpy(x):
    if isinstance(x, tf.Tensor):
        return x.numpy()
    return np.asarray(x)


def _assert_allclose(name, a, b, atol=1e-6, rtol=1e-6):
    a_np = _as_numpy(a)
    b_np = _as_numpy(b)
    np.testing.assert_allclose(a_np, b_np, atol=atol, rtol=rtol)
    print(f"{name}: OK (shape={a_np.shape})")


In [6]:
# Build both implementations
orig_lgn = lgn_module.LGN(
    row_size=ROW_SIZE,
    col_size=COL_SIZE,
    data_dir=DATA_DIR,
    n_input=N_INPUT,
    dtype=DTYPE,
)
new_lgn = LGNNumbaNumpyInit(
    row_size=ROW_SIZE,
    col_size=COL_SIZE,
    data_dir=DATA_DIR,
    n_input=N_INPUT,
    dtype=DTYPE,
)

# Attribute equivalence
_assert_allclose('x', orig_lgn.x, new_lgn.x)
_assert_allclose('y', orig_lgn.y, new_lgn.y)
_assert_allclose('non_dominant_x', orig_lgn.non_dominant_x, new_lgn.non_dominant_x)
_assert_allclose('non_dominant_y', orig_lgn.non_dominant_y, new_lgn.non_dominant_y)
_assert_allclose('amplitude', orig_lgn.amplitude, new_lgn.amplitude)
_assert_allclose('non_dom_amplitude', orig_lgn.non_dom_amplitude, new_lgn.non_dom_amplitude)
_assert_allclose('is_composite', orig_lgn.is_composite, new_lgn.is_composite)
_assert_allclose('spontaneous_firing_rates', orig_lgn.spontaneous_firing_rates, new_lgn.spontaneous_firing_rates)
_assert_allclose('dom_temporal_kernels', orig_lgn.dom_temporal_kernels, new_lgn.dom_temporal_kernels)
_assert_allclose('non_dom_temporal_kernels', orig_lgn.non_dom_temporal_kernels, new_lgn.non_dom_temporal_kernels)
_assert_allclose('sorted_neuron_ids_indices', orig_lgn.sorted_neuron_ids_indices, new_lgn.sorted_neuron_ids_indices)

assert len(orig_lgn.gaussian_filters) == len(new_lgn.gaussian_filters)
for i, (gf_o, gf_n) in enumerate(zip(orig_lgn.gaussian_filters, new_lgn.gaussian_filters)):
    _assert_allclose(f'gaussian_filters[{i}]', gf_o, gf_n)

assert len(orig_lgn.spatial_range_indices) == len(new_lgn.spatial_range_indices)
for i, (idx_o, idx_n) in enumerate(zip(orig_lgn.spatial_range_indices, new_lgn.spatial_range_indices)):
    np.testing.assert_array_equal(np.asarray(idx_o), np.asarray(idx_n))
print('spatial_range_indices: OK')


2026-02-12 15:11:45.677474: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
2026-02-12 15:11:45.683947: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
2026-02-12 15:11:45.690170: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
2026-02-12 15:11:45.698483: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX,

x: OK (shape=(17400,))
y: OK (shape=(17400,))
non_dominant_x: OK (shape=(17400,))
non_dominant_y: OK (shape=(17400,))
amplitude: OK (shape=(17400,))
non_dom_amplitude: OK (shape=(17400,))
is_composite: OK (shape=(17400,))
spontaneous_firing_rates: OK (shape=(17400,))
dom_temporal_kernels: OK (shape=(574, 17400))
non_dom_temporal_kernels: OK (shape=(314, 17400))
sorted_neuron_ids_indices: OK (shape=(17400,))
gaussian_filters[0]: OK (shape=(7, 7, 1, 1))
gaussian_filters[1]: OK (shape=(11, 11, 1, 1))
gaussian_filters[2]: OK (shape=(13, 13, 1, 1))
gaussian_filters[3]: OK (shape=(15, 15, 1, 1))
gaussian_filters[4]: OK (shape=(19, 19, 1, 1))
gaussian_filters[5]: OK (shape=(21, 21, 1, 1))
gaussian_filters[6]: OK (shape=(23, 23, 1, 1))
gaussian_filters[7]: OK (shape=(27, 27, 1, 1))
spatial_range_indices: OK


In [7]:
# Functional equivalence on a deterministic movie
seq_len_test = 64
movie = tf.random.stateless_uniform(
    shape=(seq_len_test, ROW_SIZE, COL_SIZE, 1),
    seed=tf.constant([123, 456], dtype=tf.int32),
    dtype=DTYPE,
)

sp_o = orig_lgn.spatial_response(movie, bmtk_compat=True)
sp_n = new_lgn.spatial_response(movie, bmtk_compat=True)
_assert_allclose('spatial_response[0]', sp_o[0], sp_n[0], atol=1e-5, rtol=1e-5)
_assert_allclose('spatial_response[1]', sp_o[1], sp_n[1], atol=1e-5, rtol=1e-5)

fr_o = orig_lgn.firing_rates_from_spatial(*sp_o)
fr_n = new_lgn.firing_rates_from_spatial(*sp_n)
_assert_allclose('firing_rates', fr_o, fr_n, atol=1e-5, rtol=1e-5)

print('Functional equivalence: PASSED')


2026-02-12 15:11:49.011811: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904


spatial_response[0]: OK (shape=(64, 17400))
spatial_response[1]: OK (shape=(64, 17400))
firing_rates: OK (shape=(64, 17400))
Functional equivalence: PASSED


In [8]:
def benchmark_warm_init(cls, n_runs=5):
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = cls(
            row_size=ROW_SIZE,
            col_size=COL_SIZE,
            data_dir=DATA_DIR,
            n_input=N_INPUT,
            dtype=DTYPE,
        )
        times.append(time.perf_counter() - t0)
        gc.collect()
    return np.asarray(times, dtype=np.float64)


# Warmup constructions (exclude first-call overhead)
_ = lgn_module.LGN(row_size=ROW_SIZE, col_size=COL_SIZE, data_dir=DATA_DIR, n_input=N_INPUT, dtype=DTYPE)
_ = LGNNumbaNumpyInit(row_size=ROW_SIZE, col_size=COL_SIZE, data_dir=DATA_DIR, n_input=N_INPUT, dtype=DTYPE)

orig_warm = benchmark_warm_init(lgn_module.LGN, n_runs=5)
new_warm = benchmark_warm_init(LGNNumbaNumpyInit, n_runs=5)

print('Warm-cache __init__ benchmark (seconds):')
print(f'  Original: mean={orig_warm.mean():.4f}, std={orig_warm.std():.4f}, runs={orig_warm}')
print(f'  Numba/Numpy init: mean={new_warm.mean():.4f}, std={new_warm.std():.4f}, runs={new_warm}')
print(f'  Speedup (orig/new): {orig_warm.mean() / new_warm.mean():.3f}x')


Warm-cache __init__ benchmark (seconds):
  Original: mean=0.0729, std=0.0027, runs=[0.07452289 0.07164443 0.07061817 0.07037866 0.07738499]
  Numba/Numpy init: mean=0.1054, std=0.0045, runs=[0.09979219 0.11033395 0.10112315 0.10502754 0.11062272]
  Speedup (orig/new): 0.692x


In [9]:
def _resolve_data_abs(data_dir):
    p = Path(data_dir)
    if p.is_absolute():
        return p
    return repo_root / data_dir


source_data_abs = _resolve_data_abs(DATA_DIR)
source_tf_data = source_data_abs / 'tf_data'

# Determine n_units used by this setup
n_units = int(orig_lgn.x.shape[0])
fname_cells = f'lgn_full_col_cells_{COL_SIZE}x{ROW_SIZE}.csv'
fname_spont = f'spontaneous_firing_rates_{COL_SIZE}x{ROW_SIZE}_n{n_units}.pkl'
fname_temp = f'temporal_kernels_{COL_SIZE}x{ROW_SIZE}_n{n_units}.pkl'
fname_spatial = f'spatial_kernels_{COL_SIZE}x{ROW_SIZE}_n{n_units}.pkl'
required = [fname_cells, fname_spont, fname_temp]

missing = [f for f in required if not (source_tf_data / f).exists()]
if missing:
    print('Skipping cold spatial benchmark; required files are missing:')
    for f in missing:
        print('  -', f)
    can_run_cold = False
else:
    can_run_cold = True


def benchmark_cold_spatial_init(cls, n_runs=3):
    """
    Benchmark __init__ when spatial cache is absent.
    We copy required cached files to a temp data_dir and force spatial rebuild.
    """
    times = []
    for _ in range(n_runs):
        run_dir = Path(tempfile.mkdtemp(prefix='lgn_init_bench_'))
        run_tf = run_dir / 'tf_data'
        run_tf.mkdir(parents=True, exist_ok=True)

        for f in required:
            shutil.copy2(source_tf_data / f, run_tf / f)

        spatial_path = run_tf / fname_spatial
        if spatial_path.exists():
            spatial_path.unlink()

        t0 = time.perf_counter()
        _ = cls(
            row_size=ROW_SIZE,
            col_size=COL_SIZE,
            data_dir=str(run_dir),
            n_input=N_INPUT,
            dtype=DTYPE,
        )
        times.append(time.perf_counter() - t0)
        shutil.rmtree(run_dir, ignore_errors=True)
        gc.collect()

    return np.asarray(times, dtype=np.float64)


if can_run_cold:
    # One warmup run per class (outside timing)
    _ = benchmark_cold_spatial_init(lgn_module.LGN, n_runs=1)
    _ = benchmark_cold_spatial_init(LGNNumbaNumpyInit, n_runs=1)

    orig_cold = benchmark_cold_spatial_init(lgn_module.LGN, n_runs=3)
    new_cold = benchmark_cold_spatial_init(LGNNumbaNumpyInit, n_runs=3)

    print('Cold spatial-cache __init__ benchmark (seconds):')
    print(f'  Original: mean={orig_cold.mean():.4f}, std={orig_cold.std():.4f}, runs={orig_cold}')
    print(f'  Numba/Numpy init: mean={new_cold.mean():.4f}, std={new_cold.std():.4f}, runs={new_cold}')
    print(f'  Speedup (orig/new): {orig_cold.mean() / new_cold.mean():.3f}x')


Computing spatial kernels...
Caching spatial kernels...
Computing spatial kernels...
Caching spatial kernels...
Computing spatial kernels...
Caching spatial kernels...
Computing spatial kernels...
Caching spatial kernels...
Cold spatial-cache __init__ benchmark (seconds):
  Original: mean=0.0920, std=0.0065, runs=[0.08958889 0.08552832 0.10081908]
  Numba/Numpy init: mean=0.0997, std=0.0072, runs=[0.09094596 0.10862055 0.0995612 ]
  Speedup (orig/new): 0.922x


## Notes

- Warm-cache results represent day-to-day startup when caches already exist.
- Cold spatial-cache results isolate the initialization branch that rebuilds spatial kernel bookkeeping.
- First-call Numba compilation overhead is expected and intentionally excluded from timed runs via warmup.
